# Khojki: longer controlled experiments
The 300-step control improved development CER from 7.65% to 6.91%. Extend it to 3000 steps and compare bold replay alone and a reduced 10% word mixture. Each run is bounded to ten minutes; GPU training is expected to take roughly two minutes. Fresh test lines remain excluded. Save, verify and disconnect. Leave at least 75 compute units.


In [ ]:
from pathlib import Path
import sys,os,json,time,zipfile,hashlib,subprocess,shutil
BUNDLE=Path('/content/khojki-long.zip')
assert BUNDLE.exists(),'Upload prepared bundle via Files panel.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest()=='33edfe68141e78b6d9f247046fd953605b1e947121c1ecd1c1b4f8ec91333210'
ROOT=Path('/content/khojki-ocr')
with zipfile.ZipFile(BUNDLE) as z:z.extractall('/content')
for n,h in json.loads((ROOT/'cloud-manifest.json').read_text()).items():assert hashlib.sha256((ROOT/n).read_bytes()).hexdigest()==h,n
subprocess.run([sys.executable,'-m','pip','install','-q','freetype-py','uharfbuzz','scipy','opencv-python-headless'],check=True,timeout=180)
os.environ['PYTHONPATH']=str(ROOT/'src')
import torch
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0))
ANN=ROOT/'data/round2/train-dev.json';MODEL=ROOT/'models/printed-khojki.pt';EXP=ROOT/'experiments'
assert all(r['split']!='test' for r in json.loads(ANN.read_text()))
def run(args,log,timeout=600):
 with log.open('w') as f:p=subprocess.run([sys.executable,*map(str,args)],stdout=f,stderr=subprocess.STDOUT,timeout=timeout)
 print(log.name,log.read_text()[-1600:]);p.check_returncode()
print('Fresh held-out test excluded. Three bounded 3000-step runs; checkpoints every 50 steps.')


In [ ]:
configs=[
 ('long-control',EXP/'round2-control/last.pt',['--resume']),
 ('long-bold',MODEL,['--bold-replay']),
 ('long-word10',MODEL,['--bold-replay','--word-fraction','0.1'])]
for name,checkpoint,extra in configs:
 folder=EXP/name;folder.mkdir(exist_ok=True)
 if name=='long-control' and not (folder/'best.pt').exists():shutil.copy2(EXP/'round2-control/best.pt',folder/'best.pt')
 if not (folder/'history.json').exists():
  run([ROOT/'scripts/finetune.py','--checkpoint',checkpoint,'--annotations',ANN,'--steps','3000','--lr','0.0001','--seed','137','--layout','robust','--device','cuda','--output',name,*extra],EXP/(name+'.log'))
 else:print('Saved results already exist; no automatic retraining:',name)
summary={}
for name,_,_ in configs:
 h=json.loads((EXP/name/'history.json').read_text());best=min(h,key=lambda x:x['errors'])
 summary[name]={k:v for k,v in best.items() if k!='lines'}
(EXP/'long-summary.json').write_text(json.dumps(summary,indent=2))
print('SUMMARY',json.dumps(summary,indent=2))


In [ ]:
from IPython.display import display,HTML
import base64
EXPORTS=[]
for name,_,_ in configs:
 target=Path('/content')/(name+'-results.zip')
 with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
  for p in (EXP/name).glob('*'):
   if p.is_file():z.write(p,str(p.relative_to(ROOT)))
  for p in [EXP/(name+'.log'),EXP/'long-summary.json']:z.write(p,str(p.relative_to(ROOT)))
 h=hashlib.sha256(target.read_bytes()).hexdigest()
 display(HTML('<a download="'+target.name+'" href="data:application/zip;base64,'+base64.b64encode(target.read_bytes()).decode()+'">Download '+name+' results</a>'))
 EXPORTS.append({'name':target.name,'bytes':target.stat().st_size,'sha256':h})
print('EXPORT_MANIFEST',json.dumps(EXPORTS))
